# Cell Quality Check + Chirp + Cell Typing

contact: ron.w.ditullio@gmail.com based on Guilhelm's guilhelm-dev branch

execution time:

Tested on Ubuntu 24.04.2 LTS (32 cores, 188 GiB RAM, Intel(R) Core(TM) i9-14900K)

In [1]:
%load_ext autoreload
%autoreload 2

# import packages: primary
import os
import numpy as np
import matplotlib.pyplot as plt

# import packages for clustering #2026-01-21 RWD: sklearn is not in the environment by default.  Maybe add to yaml?

# import custom packages
import params
from utils import chirp as analysis
import utils
import gc


-------- Creating all paths ---------

- phy (.GUI): using your override
    /media/idv-s8/SSD Storage/20260729_VideoLSTA_calib/RAW_DATA/20260729_meas_00_SWN_30Hz/20260729_meas_00_SWN_30Hz.GUI
- "output" path already exists
- "triggers" path already exists


/home/idv-s8/anaconda3/envs/std_analysis_pipeline/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Cell 1: Load triggers and spikes + select chirp type !!!!

In [2]:
old = False  # False = new 50 Hz chirp ; True = old 2p-room chirp

cells, spike_times, stim_onsets, vec_keys, check_directory, CT_directory, old = (
    analysis.get_all_inputs_for_chirp_analysis(params, old)
)

Which of the following is the chirp recording: 
	0 --> 20260729_meas_00_SWN_30Hz
	1 --> 20260729_meas_01_SWN_30Hz
	2 --> 20260729_meas_02_chirp_50Hz
	3 --> 20260729_meas_03_DG_2ST_10rep_50Hz
	4 --> 20260729_meas_04_barcode_3dir_50Hz
	5 --> 20260729_meas_05_ RMO_Pert_CALIB_baseline_40Hz
	6 --> 20260729_meas_06_RMO_Pert_CALIB_c15_f04_40Hz
	7 --> 20260729_meas_07_RMO_Pert_CALIB_c15_f12_40Hz
	8 --> 20260729_meas_08_RMO_Pert_CALIB_c20_f04_40Hz_2026
	9 --> 20260729_meas_09_RMO_Pert_CALIB_c20_f12_40Hz
	10 --> 20260729_meas_10_RMO_Pert_CALIB_baseline_40Hz
	11 --> 20260729_meas_11_RMO_Pert_CALIB_c25_f04_40Hz
	12 --> 20260729_meas_12_RMO_Pert_CALIB_c25_f12_40Hz_2026
	13 --> 20260729_meas_13_RMO_Pert_CALIB_c30_f04_40Hz
	14 --> 20260729_meas_14_RMO_Pert_CALIB_c30_f12_40Hz
	15 --> 20260729_meas_15_RMO_Pert_CALIB_baseline_40Hz
Selected recording: 20260729_meas_02_chirp_50Hz


Selected recording : 20260729_meas_02_chirp_50Hz 

Creating analysis directory...
/media/idv-s8/SSD Storage/20260729_VideoLST

## Cell 1b: Refractory-period-violation (RPV) filter

Cells whose RPV exceeds the guideline are dropped from the clustering set (they can still appear in the rasters/plots above). Requires the phy sorting output.

In [3]:
# Cells whose refractory-period-violation (RPV) rate exceeds the guideline are excluded
# from clustering below. (RPV needs the phy sorting output in params.phy_directory.)
rpv_len = 2  # ms: inter-spike intervals below this count as refractory violations
rpv_threshold = 0.5  # %: maximum acceptable RPV rate

cell_rpvs = utils.get_cell_rpvs(
    cells,
    params.phy_directory,
    rpv_len,
)
good_cells = [c for c in cells if cell_rpvs[c]["rpv"] < rpv_threshold]
print(
    f"{len(good_cells)}/{len(cells)} cells pass the RPV filter (RPV < {rpv_threshold}%)"
)

213/234 cells pass the RPV filter (RPV < 0.5%)


## Cell 2: Chirp rasters

In [4]:
cell_data = analysis.compute_chirp_rasters(
    cells, spike_times, stim_onsets, vec_keys, old=old, n_bins=800, n_bins_small=16000
)

Extracting cells responses to Chirp stimulus



Extraction: 100%|██████████| 234/234 [00:00<00:00, 392.43it/s]


## Cell 3: Plot Chirp rasters

With added spatial STA in order to be able to use this plot alone to do cell selection for clustering

#### <center><i>REQUIRES CELL 2 RUN AND CHECKERBOARD ANALYSIS</center>

In [5]:
analysis.plot_chirp_rasters(cells, cell_data, CT_directory, check_directory)


Stimulus files in ./ResourcesAndTools/StandardVec:
    0 : 01_DG_50hz_10reps_8dir_2sT_std.vec
    1 : 20250512_4_SWN_48pixCh_6pixShift_30Hz_MEA2.vec
    2 : 20250512_4_SWN_48pixCh_6pixShift_30Hz_MEA2_std.vec
    3 : 4squares_std.vec
    4 : DG_50hZ_8reps_8dir_2sT.vec
    5 : DG_50hZ_8reps_8dir_2sT_std.vec
    6 : Euler_50Hz_20reps_1024x768pix_std.vec   <- default
Using stimulus file: Euler_50Hz_20reps_1024x768pix_std.vec

Saving Chirp raster plots in : /media/idv-s8/SSD Storage/20260729_VideoLSTA_calib/Analysis/CellTyping_Analysis_rec_2/Chirp_rasters+STA 



100%|██████████| 234/234 [00:39<00:00,  5.97it/s]

--- Cell Done ---


## Cell 4a: Select cells with a good STA

Keep the cells that have a **well-defined receptive field (STA)**. Each cell's checkerboard
STA figure is shown one at a time — type `Yes` to keep it. Pre-fill `good_sta_cells` to skip
the review; if a selection was saved earlier it is reused automatically.

In [ ]:
# ---- Select cells with a good STA -------------------------------------
# Leave good_sta_cells = [] to review each cell's STA figure (type Yes to keep), or pre-fill
# it to skip the review. A previously saved selection is reused (never lost); the result is
# saved automatically. To redo the review, pre-fill the list or delete the selection file.
good_sta_cells = []
selected_cells_sta = utils.select_cells_by_sta(
    good_cells, check_directory, CT_directory, params, preselected=good_sta_cells
)
print(f"{len(selected_cells_sta)} cells with a good STA")

## Cell 4b: Select cells with a good chirp

Now keep the cells with a **clear chirp response** (its raster / PSTH). A cell is used for
clustering only if it passes **both** steps — good STA *and* good chirp — so this block also
combines the two selections and saves them.

In [ ]:
# ---- Select cells with a good chirp response --------------------------
# Same idea as the STA step, using each cell's chirp figure. Each step SAVES its own list
# (updating only its part), so running one never erases the other's selection.
good_chirp_cells = []
selected_cells_chirp = utils.select_cells_by_chirp(
    good_cells, CT_directory, params, preselected=good_chirp_cells
)
print(f"{len(selected_cells_chirp)} cells with a good chirp")

# Final selection = good STA AND good chirp (read from the saved file, so it is correct even
# if the STA step above was run in a previous session).
selected_cells = utils.load_cell_selection(CT_directory, params)["selected_cells"]
print(f"{len(selected_cells)} cells selected for clustering (good STA AND good chirp)")

## Cell 4c: Manually adjust the selection (optional)

Add or remove cells from the good-STA / good-chirp lists by hand — e.g. if the interactive
review misjudged a cell. This **edits the saved selection in place** (it never erases it), so
lists you leave empty change nothing. Run it as many times as you like; run it before the DOS
split below, since that split uses the resulting `selected_cells`.

In [ ]:
# Cells to add / remove (leave a list empty to change nothing). This edits the SAVED
# selection non-destructively — it never clears the lists.
sta_add = []  # add to the good-STA list
sta_remove = []  # remove from the good-STA list
chirp_add = []  # add to the good-chirp list
chirp_remove = []  # remove from the good-chirp list
remove_from_both = []  # remove from BOTH lists at once (a cell you no longer want at all)

selected_cells_sta, selected_cells_chirp, selected_cells = utils.edit_cell_selection(
    CT_directory,
    params,
    sta_add=sta_add,
    sta_remove=sta_remove,
    chirp_add=chirp_add,
    chirp_remove=chirp_remove,
    remove_from_both=remove_from_both,
)

## Cell 4d: Select orientation-/direction-selective (DOS) cells

Uses the DG analysis (notebook 3). Cells that are **orientation-selective or direction-selective** — together called **DOS** — are split off from the rest, so the two groups are cell-typed separately below. Set `dos_cells` manually, or leave it empty to pick interactively from the DG plots (or reload a saved selection).

In [ ]:
# The DG analysis (notebook 3) must have been run for this experiment.
DG_directory = utils.find_analysis_directory(params.output_directory, "DG")

dos_cells = []  # set manually, e.g. [12, 45, 78] ; empty = interactive (or reload a saved selection)

dos_cells, non_dos_cells = utils.select_dos_cells(
    selected_cells, dos_cells, DG_directory, CT_directory, params
)

## Cell 5: Cell typing — cluster non-DOS and DOS cells separately

Each group is clustered on its own with Agglomerative Clustering: first the non-DOS cells, then the DOS (orientation-/direction-selective) ones. The two groups can use different parameters.

-you want to move the distance threshold until you have roughly 20 clusters for non-DOS and 10 clusters for DOS <br>
-you want to have a number of PCs that cumulatively can explain around 80% of the variance in the dataset <br>
-you can decide how many components of the STA to choose in the clustering (usually 2 if the checkerboard recording is reliable and 1 otherwise)

#### <center><i>REQUIRES CELL 2, CELL 4 AND CELL 5 RUN </center>

In [ ]:
# --- Cluster the non-DOS cells ---
# Move dist_thres to adapt the dendrogram cut / number of clusters.
dist_thres_non_dos = 32
n_components_psth_non_dos = 25  # PCs from the chirp PSTH (aim for ~80% variance)
n_components_sta_tc_non_dos = (
    2  # PCs from the STA temporal course (2 if checkerboard reliable, else 1)
)
sparse = False


# run_cell_typing_AC drops cells with a flat/NaN PSTH or STA and returns the kept list,
# so non_dos_cells is updated to exactly the cells that were clustered.
psth_z_non_dos, sta_results, model_non_dos, non_dos_cells = utils.run_cell_typing_AC(
    dist_thres_non_dos,
    n_components_psth_non_dos,
    n_components_sta_tc_non_dos,
    cell_data,
    non_dos_cells,
    check_directory,
    sparse,
)

### Cluster the DOS (orientation-/direction-selective) cells

In [ ]:
# --- Cluster the DOS cells (their own parameters) ---
dist_thres_dos = 25
n_components_psth_dos = 4
n_components_sta_tc_dos = 2

# dos_cells is likewise updated to the cells actually clustered (flat/NaN ones dropped).
psth_z_dos, _, model_dos, dos_cells = utils.run_cell_typing_AC(
    dist_thres_dos,
    n_components_psth_dos,
    n_components_sta_tc_dos,
    cell_data,
    dos_cells,
    check_directory,
    sparse,
)

## Cell 6: Merge the two labellings

DOS cluster IDs start right after the non-DOS ones, so the two never collide. Each clustered cell gets `cell_data[cell]["type"]` (cluster ID) and `cell_data[cell]["dos"]` (True/False — orientation- or direction-selective).

In [ ]:
# DOS cluster IDs start after the non-DOS ones so the two labellings don't collide.
# non_dos_cells / dos_cells are the cells that were actually clustered (run_cell_typing_AC
# may have dropped some); every other cell is left "Not assigned".
n_non_dos_clusters = len(np.unique(model_non_dos.labels_))

for i, cell in enumerate(non_dos_cells):
    cell_data[cell]["type"] = int(model_non_dos.labels_[i])
    cell_data[cell]["dos"] = False
for i, cell in enumerate(dos_cells):
    cell_data[cell]["type"] = int(model_dos.labels_[i]) + n_non_dos_clusters
    cell_data[cell]["dos"] = True

clustered = set(non_dos_cells) | set(dos_cells)
for cell in cell_data:
    if cell not in clustered:
        cell_data[cell]["type"] = "Not assigned"
        cell_data[cell]["dos"] = None

# Combined views for the downstream cells (cross-corr, summary): only the clustered
# cells, in cell_data order, with psth_z aligned to selected_cells.
selected_cells = [c for c in cell_data if c in clustered]
psth_z = np.vstack(
    [
        psth_z_non_dos[non_dos_cells.index(c)]
        if c in non_dos_cells
        else psth_z_dos[dos_cells.index(c)]
        for c in selected_cells
    ]
)

n_dos_clusters = len(np.unique(model_dos.labels_))
print(
    f"non-DOS clusters: {n_non_dos_clusters} | DOS clusters: {n_dos_clusters} "
    f"| total: {n_non_dos_clusters + n_dos_clusters} | clustered cells: {len(selected_cells)}"
)

## Cell 7: Create a summary figure for each cluster type

#### <center><i>REQUIRES CELL 6 RUN </center>

In [ ]:
# selected_cells = the cells actually clustered (dropped cells excluded).
utils.create_cluster_summary_figure(
    cell_data, selected_cells, psth_z, sta_results, params, CT_directory, old
)

plt.close("all")

gc.collect()

## Cell 8: When satisfied with the clustering, save data

In [ ]:
exp = params.exp
fsave = os.path.join(CT_directory, "{}_cell_typing_data".format(exp))
utils.save_obj(cell_data, fsave)

## (Optional) Plot a hand-made cluster

Group any cells you like into a named cluster and get the same mosaic figure. Set the
cluster name and the cell list below. The cells should be among the clustered cells
(`selected_cells`); the figure is saved as `Cluster_<name>.png` in the Cell_typing folder.

In [ ]:
handmade_cluster_name = ""  # name of the group (also the figure file name)
handmade_cell_list = []  # e.g. [208, 209, 210] ; the cells to group together

utils.plot_handmade_cluster(
    handmade_cluster_name,
    handmade_cell_list,
    cell_data,
    selected_cells,
    psth_z,
    sta_results,
    params,
    CT_directory,
    old,
)